<a href="https://colab.research.google.com/github/jbaldo1305/CAPSTONE2/blob/main/CAPSTONE3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Dialogue Summarization with T5 on the SAMSum Dataset

## Business Problem
Acme Communications wants to reduce information overload in group chats by automatically generating concise summaries of long conversations. This project develops a proof-of-concept dialogue summarization model using the SAMSum dataset, which contains messenger-style conversations paired with human-written summaries.

## Project Goal
The goal of this project is to fine-tune a transformer-based text summarization model that can generate concise and meaningful summaries from multi-speaker dialogue conversations.

## Why This Matters
A successful summarization model could help users catch up faster, reduce cognitive overload, and improve engagement with the messaging platform.

In [ ]:
import torch
print(torch.cuda.is_available())

True


In [ ]:
!pip install transformers datasets evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.9 MB/s eta 0:00:00


##IMPORTING LIBRARIES

In [ ]:
from datasets import load_dataset
from transformers import T5Tokenizer, T5ForConditionalGeneration, Trainer, TrainingArguments
import evaluate
import torch
import numpy as np
import re

## LOADING THE DATASET

The SAMSum dataset is loaded directly from Hugging Face. It contains short messenger-style conversations and corresponding human-written summaries. The main columns used in this project are:

- `dialogue`: the input conversation
- `summary`: the target human-written summary

In [ ]:
dataset = load_dataset("knkarthick/samsum")
dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 14731
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 818
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 819
    })
})

##INSPECTING DATASET

In [ ]:
print(dataset["train"][0])
print("Train size:", len(dataset["train"]))
print("Validation size:", len(dataset["validation"]))
print("Test size:", len(dataset["test"]))

{'id': '13818513', 'dialogue': "Amanda: I baked  cookies. Do you want some?\nJerry: Sure!\nAmanda: I'll bring you tomorrow :-)", 'summary': 'Amanda baked cookies and will bring Jerry some tomorrow.'}
Train size: 14731
Validation size: 818
Test size: 819


The dataset contains three main fields: id, dialogue, and summary. The dialogue column contains the conversation text, while the summary column contains the reference human-written summary.

## SUBSET SELECTION
Because transformer training can be computationally expensive in Google Colab, a subset of the dataset is used for this proof-of-concept. This keeps runtime manageable while still allowing meaningful evaluation.

In [ ]:
train_data = dataset["train"].select(range(2000))
val_data = dataset["validation"].select(range(300))
test_data = dataset["test"].select(range(300))

print("Subset train size:", len(train_data))
print("Subset validation size:", len(val_data))
print("Subset test size:", len(test_data))

Subset train size: 2000
Subset validation size: 300
Subset test size: 300


Due to compute limits in Google Colab, a subset of the dataset was used for rapid experimentation and proof-of-concept modeling.

## TEXT CLEANING

Although SAMSum is already relatively clean, light preprocessing was applied to reduce noise and improve consistency before training.

The cleaning steps included:
- trimming leading and trailing whitespace
- normalizing repeated spaces and line breaks
- removing extra spaces before punctuation
- replacing media placeholders such as `<file_gif>`, `<photo>`, and `<video>` with a neutral token `[media]`

Speaker names, punctuation, and dialogue structure were preserved because they are important in dialogue summarization and help the model understand who said what.



In [ ]:
def clean_text(text):
    text = text.strip()
    text = re.sub(r"\s+", " ", text)                 # normalize repeated whitespace
    text = re.sub(r"\s+([?.!,;:])", r"\1", text)     # remove space before punctuation

    # replace media/file placeholders with a neutral marker
    text = text.replace("<file_gif>", "[media]")
    text = text.replace("<photo>", "[media]")
    text = text.replace("<video>", "[media]")
    text = text.replace("<file_photo>", "[media]")
    text = text.replace("<file_video>", "[media]")

    return text

In [ ]:
def clean_dataset(example):
    example["dialogue"] = clean_text(example["dialogue"])
    example["summary"] = clean_text(example["summary"])
    return example

In [ ]:
train_data = train_data.map(clean_dataset)
val_data = val_data.map(clean_dataset)
test_data = test_data.map(clean_dataset)

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

In [ ]:
print("Cleaned dialogue example:\n")
print(train_data[0]["dialogue"])
print("\nCleaned summary example:\n")
print(train_data[0]["summary"])

Cleaned dialogue example:

Amanda: I baked cookies. Do you want some? Jerry: Sure! Amanda: I'll bring you tomorrow:-)

Cleaned summary example:

Amanda baked cookies and will bring Jerry some tomorrow.


## MODEL SELECTION

T5-small was selected as the primary model for this project. T5 is a text-to-text transformer, which makes it well suited for summarization tasks because both the input and output are text.

BART was also considered, but T5 was chosen for the initial implementation because it is relatively straightforward to use and computationally lighter for experimentation in Google Colab.

In [ ]:
model_name = "t5-small"
tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

##TOKENIZATION FUNCTION

The dialogue text is converted into token IDs that the model can process. Because T5 uses a text-to-text framework, each dialogue is prefixed with `summarize:` so the model knows which task to perform.

The target summaries are also tokenized. Padding tokens in the labels are replaced with `-100` so they are ignored during loss calculation, which helps training behave correctly.




In [ ]:
def preprocess_function(examples):
    inputs = ["summarize: " + dialogue for dialogue in examples["dialogue"]]
    targets = [summary for summary in examples["summary"]]

    model_inputs = tokenizer(
        inputs,
        max_length=512,
        truncation=True,
        padding="max_length"
    )

    labels = tokenizer(
        text_target=targets,
        max_length=128,
        truncation=True,
        padding="max_length"
    )

    label_ids = labels["input_ids"]
    label_ids = [
        [(token if token != tokenizer.pad_token_id else -100) for token in label]
        for label in label_ids
    ]

    model_inputs["labels"] = label_ids
    return model_inputs

In [ ]:
train_tokenized = train_data.map(preprocess_function, batched=True)
val_tokenized = val_data.map(preprocess_function, batched=True)
test_tokenized = test_data.map(preprocess_function, batched=True)

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

##TRAINING ARGUMENTS
The model is fine-tuned using a modest training configuration suitable for Google Colab. A small batch size and limited number of epochs are used to balance runtime and performance.

In [ ]:
training_args = TrainingArguments(
    output_dir="./t5_samsum_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_steps=50,
    save_total_limit=1,
    report_to="none"
)

##TRAIN MODEL

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized
)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,2.237672,1.968765
2,2.117608,1.944935


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1000, training_loss=2.2814410018920896, metrics={'train_runtime': 229.6523, 'train_samples_per_second': 17.418, 'train_steps_per_second': 4.354, 'total_flos': 541367205888000.0, 'train_loss': 2.2814410018920896, 'epoch': 2.0})

## QUALITIVE EVALUATION

To inspect model quality beyond numeric metrics, several generated summaries are compared against the human-written summaries from the test set.

In [ ]:
model.eval()

for i in range(3):
    sample = test_data[i]
    input_text = "summarize: " + sample["dialogue"]

    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_length=60,
        num_beams=4,
        early_stopping=True
    )

    predicted_summary = tokenizer.decode(outputs[0], skip_special_tokens=True)

    print(f"Example {i+1}")
    print("DIALOGUE:\n", sample["dialogue"])
    print("\nACTUAL SUMMARY:\n", sample["summary"])
    print("\nPREDICTED SUMMARY:\n", predicted_summary)
    print("\n" + "="*100 + "\n")

Example 1
DIALOGUE:
 Hannah: Hey, do you have Betty's number? Amanda: Lemme check Hannah: [media] Amanda: Sorry, can't find it. Amanda: Ask Larry Amanda: He called her last time we were at the park together Hannah: I don't know him well Hannah: [media] Amanda: Don't be shy, he's very nice Hannah: If you say so.. Hannah: I'd rather you texted him Amanda: Just text him 🙂 Hannah: Urgh.. Alright Hannah: Bye Amanda: Bye bye

ACTUAL SUMMARY:
 Hannah needs Betty's number but Amanda doesn't have it. She needs to contact Larry.

PREDICTED SUMMARY:
 Amanda has Betty's number. He called her last time we were at the park together.


Example 2
DIALOGUE:
 Eric: MACHINE! Rob: That's so gr8! Eric: I know! And shows how Americans see Russian;) Rob: And it's really funny! Eric: I know! I especially like the train part! Rob: Hahaha! No one talks to the machine like that! Eric: Is this his only stand-up? Rob: Idk. I'll check. Eric: Sure. Rob: Turns out no! There are some of his stand-ups on youtube. Eric:

In [ ]:
!pip install rouge_score

##EVALUATING WITH ROUGE

Model performance is evaluated using ROUGE metrics:
- **ROUGE-1**: unigram overlap
- **ROUGE-2**: bigram overlap
- **ROUGE-L**: longest common subsequence overlap

ROUGE-2 is especially useful here because it measures phrase-level similarity and helps assess whether the model preserves contextual relationships between words.

In [ ]:
rouge = evaluate.load("rouge")

predictions = []
references = []

for i in range(100):
    sample = test_data[i]
    input_text = "summarize: " + sample["dialogue"]

    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_length=60,
        num_beams=4,
        early_stopping=True
    )

    predicted_summary = tokenizer.decode(outputs[0], skip_special_tokens=True)

    predictions.append(predicted_summary)
    references.append(sample["summary"])

results = rouge.compute(predictions=predictions, references=references)
results

{'rouge1': np.float64(0.37132609438621567),
 'rouge2': np.float64(0.14296796639320203),
 'rougeL': np.float64(0.3032993922942565),
 'rougeLsum': np.float64(0.30335258568161017)}

## Results Interpretation

The model achieved improved ROUGE scores after applying preprocessing, training adjustments, and improved generation strategies.

- **ROUGE-1 (0.37)** indicates that the model is effectively capturing important words from the reference summaries.
- **ROUGE-2 (0.14)** reflects moderate success in preserving two-word phrase relationships, suggesting that the model is beginning to capture contextual meaning rather than just individual keywords.
- **ROUGE-L (0.30)** demonstrates that the generated summaries maintain a reasonable level of sentence structure and coherence.

These results represent a meaningful improvement over the initial model, which produced empty outputs due to preprocessing and training issues. After correcting the preprocessing pipeline and improving generation parameters (beam search), the model was able to generate coherent and relevant summaries.

Given that the model was trained on a subset of the dataset and only for a small number of epochs, these results are considered strong for a proof-of-concept implementation.

## Business Conclusion

This project demonstrates the feasibility of implementing an AI-powered dialogue summarization system to reduce information overload in messaging platforms.

The model successfully generates concise summaries that capture key elements of multi-speaker conversations, allowing users to quickly understand the main points without reading the full dialogue.

From a business perspective, this solution could:

- reduce the time users spend catching up on long conversations
- improve user engagement by making content easier to digest
- enhance the overall user experience within the messaging platform
- introduce valuable AI-driven functionality that differentiates the product

While the current model is not production-ready, the results indicate that transformer-based summarization models such as T5 can provide meaningful value even with limited training data and computational resources.

This proof-of-concept supports further investment in more advanced models and larger-scale training to deploy summarization features in real-world applications.

## Limitations and Next Steps

This project was completed as a proof-of-concept using a subset of the SAMSum dataset and a lightweight transformer model (T5-small). While the model achieved meaningful ROUGE scores and produced coherent summaries, there are several limitations to consider.

First, the model was trained on a relatively small subset of the data and for a limited number of epochs, which restricts its ability to fully learn complex dialogue patterns. Additionally, dialogue summarization is inherently challenging due to informal language, multiple speakers, and implicit context that may not always be captured in short summaries.

Another limitation is that evaluation was based primarily on ROUGE metrics, which measure word and phrase overlap but do not fully capture semantic understanding or summary quality from a human perspective.

Future improvements could include:

- training on a larger portion of the dataset
- increasing the number of training epochs
- experimenting with larger models such as T5-base or BART
- tuning generation parameters further (e.g., beam width, length penalties)
- incorporating human evaluation alongside ROUGE metrics

Despite these limitations, the project demonstrates that transformer-based models can effectively summarize conversational data and provides a strong foundation for further development and real-world application.